In [1]:
import torch
import torch.nn.functional as F
import time

def profile_attention_mechanisms():
    # 시퀀스 길이는 극단적으로 길고, 차원은 압축된 가혹 환경 시뮬레이션
    N = 15000  # Sequence Length (예: 초고해상도 이미지 패치 또는 초대형 문서)
    D = 64     # Hidden Dimension
    B = 1      # Batch Size

    Q = torch.randn(B, N, D).cuda()
    K = torch.randn(B, N, D).cuda()
    V = torch.randn(B, N, D).cuda()

    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    # ----------------------------------------------------
    # 1. Standard Self-Attention 프로파일링
    # ----------------------------------------------------
    mem_start_std = torch.cuda.memory_allocated()
    start_time = time.time()

    # QK^T 연산 -> [B, N, N] 어텐션 맵 생성 (메모리 폭발의 주범)
    attn_weights = torch.matmul(Q, K.transpose(-2, -1)) / (D ** 0.5)
    attn_probs = F.softmax(attn_weights, dim=-1)
    out_std = torch.matmul(attn_probs, V)

    torch.cuda.synchronize()
    std_time = time.time() - start_time
    mem_end_std = torch.cuda.memory_allocated()
    std_mem_used = (mem_end_std - mem_start_std) / (1024 ** 2) # MB 전환

    del attn_weights, attn_probs, out_std
    torch.cuda.empty_cache()

    # ----------------------------------------------------
    # 2. Effective Attention 프로파일링
    # ----------------------------------------------------
    mem_start_eff = torch.cuda.memory_allocated()
    start_time = time.time()

    # 각 행렬의 독립적 정규화 선행
    q_reg = F.softmax(Q, dim=-1)
    k_reg = F.softmax(K, dim=-2) # 행/열 기준 분할 정규화

    # 결합법칙 우회 연산: K^T * V를 먼저 수행 -> [B, D, D] 컨텍스트 맵 생성
    context_matrix = torch.matmul(k_reg.transpose(-2, -1), V)
    # 최종 결과 도출: Q * (K^T * V) -> [B, N, D]
    out_eff = torch.matmul(q_reg, context_matrix)

    torch.cuda.synchronize()
    eff_time = time.time() - start_time
    mem_end_eff = torch.cuda.memory_allocated()
    eff_mem_used = (mem_end_eff - mem_start_eff) / (1024 ** 2)

    print("=== [실험 3] 연산 패러다임 시프트 결과 ===")
    print(f"[Standard Self-Attention] 시간: {std_time:.6f}s | 추가 메모리: {std_mem_used:.2f} MB")
    print(f"[Effective Attention]    시간: {eff_time:.6f}s | 추가 메모리: {eff_mem_used:.2f} MB")
    print(f"▶ 메모리 절감 효율: {std_mem_used / max(eff_mem_used, 1e-5):.1f}배 개선 완료")

if torch.cuda.is_available():
    profile_attention_mechanisms()
else:
    print("이 실험은 CUDA 가속 환경(GPU) 전용 프로파일러 코드로 작성되었습니다.")

=== [실험 3] 연산 패러다임 시프트 결과 ===
[Standard Self-Attention] 시간: 0.233416s | 추가 메모리: 1729.29 MB
[Effective Attention]    시간: 0.043576s | 추가 메모리: 11.00 MB
▶ 메모리 절감 효율: 157.2배 개선 완료
